## Step 7: **Launch the Gradio Interface**

In [ ]:
# Fix langserve + httpx compatibility — requires runtime restart
!pip install -q "httpx==0.27.2" "langserve==0.3.0" "sse-starlette>=1.6.1"

import os
print("\u2705 langserve dependencies installed. Restarting runtime...")
os.kill(os.getpid(), 9)

In [18]:
import re
import logging
import threading

# ── Suppress noisy loggers ────────────────────────────────────────────────────
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)
logging.getLogger("uvicorn.error").setLevel(logging.CRITICAL)

# ── Patch gradio_client schema parser bugs ────────────────────────────────────
import gradio_client.utils as _gcu

_orig_get_type = _gcu.get_type
def _patched_get_type(schema):
    if not isinstance(schema, dict):
        return "any"
    return _orig_get_type(schema)
_gcu.get_type = _patched_get_type

_orig_json_schema = _gcu._json_schema_to_python_type
def _patched_json_schema(schema, defs=None):
    try:
        return _orig_json_schema(schema, defs)
    except Exception:
        return "any"
_gcu._json_schema_to_python_type = _patched_json_schema

_orig_public = _gcu.json_schema_to_python_type
def _patched_public(schema, defs=None):
    try:
        return _orig_public(schema)
    except Exception:
        return "any"
_gcu.json_schema_to_python_type = _patched_public

print("✅ gradio_client patches applied.")
# ─────────────────────────────────────────────────────────────────────────────

# ── LangServe REST API (start only once per session) ─────────────────────────
try:
    from fastapi import FastAPI
    from langserve import add_routes
    import uvicorn

    _langserve_started = globals().get("_langserve_started", False)
    if not _langserve_started:
        api_app = FastAPI(
            title="Smart Contract RAG API",
            description="LangServe REST API for Smart Contract Q&A",
            version="1.0.0",
        )
        add_routes(api_app, rag_chain.as_runnable(), path="/contract-qa")

        def _run_langserve():
            uvicorn.run(api_app, host="0.0.0.0", port=8001, log_level="error")

        threading.Thread(target=_run_langserve, daemon=True).start()
        _langserve_started = True
        print("✅ LangServe API started on port 8001")
    else:
        print("✅ LangServe already running — skipped.")
except Exception as e:
    print(f"⚠️ LangServe could not start: {e}")
# ─────────────────────────────────────────────────────────────────────────────

# ========================================================
#  GRADIO UI — Smart Contract Q&A Assistant
# ========================================================

uploaded_files_registry = []


def handle_upload(files) -> str:
    """Ingest uploaded documents — clears DB first to prevent source mixing."""
    if not files:
        return "⚠️ No files were uploaded. Please select at least one file."
    vsm.clear()
    uploaded_files_registry.clear()
    results = []
    for file in files:
        fpath = file.name if hasattr(file, "name") else str(file)
        fname = Path(fpath).name
        try:
            chunks = ingester.ingest(fpath)
            vsm.add_documents(chunks)
            uploaded_files_registry.append(fname)
            results.append(f"✅ **{fname}** — {len(chunks)} chunks indexed")
        except Exception as e:
            results.append(f"❌ **{fname}** — Error: {e}")
    lines  = ["### 📥 Ingestion Complete\n"] + results
    lines += [f"\n📊 **Total chunks in database: {vsm.document_count}**"]
    lines += ["\n✨ Ready! Ask any question about the uploaded document."]
    return "\n\n".join(lines)


def load_sample_contract() -> str:
    """Load built-in AlphaToken sample — clears DB first."""
    vsm.clear()
    uploaded_files_registry.clear()
    try:
        chunks = ingester.ingest("/content/sample_smart_contract.txt")
        vsm.add_documents(chunks)
        return (
            "### ✅ Sample Contract Loaded\n\n"
            "📄 *AlphaToken (ALPHA) — Token Sale & Vesting Protocol*\n\n"
            f"📊 **{len(chunks)} chunks** indexed and ready.\n\n"
            "**Try asking:**\n"
            "- What is the total token supply?\n"
            "- What are the vesting terms for the team?\n"
            "- How does the refund policy work?\n"
            "- Which jurisdictions are restricted?"
        )
    except Exception as e:
        return f"❌ Error loading sample contract: {e}"


def clear_database() -> str:
    """Wipe the vector database entirely."""
    vsm.clear()
    uploaded_files_registry.clear()
    return (
        "### 🗑️ Database Cleared\n\n"
        "All indexed documents have been removed.\n"
        "Upload a new contract to continue."
    )


def chat(message: str, history: list) -> Tuple[list, str]:
    """Full RAG pipeline: guardrails → retrieval → generation → citations."""
    if not message or not message.strip():
        return history, ""

    if vsm.document_count == 0:
        reply = (
            "⚠️ **No documents loaded.**\n\n"
            "Please either:\n"
            "1. Upload a PDF, DOCX, or TXT contract using the left panel, or\n"
            "2. Click **Load Sample** to use the built-in demo contract."
        )
        return history + [
            {"role": "user",      "content": message},
            {"role": "assistant", "content": reply},
        ], ""

    answer, sources = rag_chain.ask(message)

    if sources and sources.strip():
        full_reply = (
            f"{answer}\n\n"
            f"---\n"
            f"#### 📚 Referenced Sources\n\n"
            f"{sources}"
        )
    else:
        full_reply = answer

    return history + [
        {"role": "user",      "content": message},
        {"role": "assistant", "content": full_reply},
    ], ""


# ── Professional Light-Theme CSS ──────────────────────────────────────────────
CSS = """
.gradio-container {
    font-family: 'Inter', 'Segoe UI', Arial, sans-serif !important;
    background: #f0f4f8 !important;
    max-width: 1380px !important;
    margin: 0 auto !important;
}
.header-box {
    background: linear-gradient(120deg, #1a1a2e 0%, #16213e 55%, #0f3460 100%);
    padding: 26px 36px;
    border-radius: 16px;
    margin-bottom: 18px;
    box-shadow: 0 6px 24px rgba(15,52,96,0.22);
    border: 1px solid rgba(255,255,255,0.07);
}
.panel-card {
    background: #ffffff !important;
    border-radius: 14px !important;
    border: 1px solid #dde3ec !important;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06) !important;
    padding: 4px 10px !important;
}
.status-md {
    background: #f0f7ff !important;
    border: 1px solid #bfdbfe !important;
    border-radius: 10px !important;
    padding: 12px 16px !important;
    font-size: 13.5px !important;
    color: #1e3a5f !important;
    min-height: 60px !important;
}
button.lg.primary {
    background: linear-gradient(135deg, #1d4ed8, #0ea5e9) !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    font-size: 15px !important;
    color: #ffffff !important;
    box-shadow: 0 2px 8px rgba(14,165,233,0.25) !important;
    transition: opacity 0.18s ease !important;
}
button.lg.primary:hover { opacity: 0.88 !important; }
button.sm.secondary {
    background: #eff6ff !important;
    border: 1px solid #93c5fd !important;
    color: #1d4ed8 !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
}
button.sm.stop {
    background: #fff1f2 !important;
    border: 1px solid #fca5a5 !important;
    color: #dc2626 !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
}
.gr-textbox textarea {
    border: 1.5px solid #93c5fd !important;
    border-radius: 10px !important;
    font-size: 14.5px !important;
    color: #1e293b !important;
    background: #f8fafc !important;
    padding: 10px 14px !important;
}
.gr-textbox textarea:focus {
    border-color: #2563eb !important;
    box-shadow: 0 0 0 3px rgba(37,99,235,0.12) !important;
    outline: none !important;
}
.gr-chatbot {
    border: 1px solid #dde3ec !important;
    border-radius: 12px !important;
    background: #f8fafc !important;
}
.gr-markdown p, .gr-markdown li {
    color: #374151 !important;
    font-size: 14px !important;
    line-height: 1.65 !important;
}
.gr-markdown code {
    background: #f1f5f9 !important;
    border-radius: 5px !important;
    padding: 2px 6px !important;
    font-size: 12.5px !important;
    color: #0f172a !important;
}
.footer-box {
    text-align: center;
    padding: 14px 0 4px 0;
    font-size: 0.83em;
    color: #64748b;
    border-top: 1px solid #e2e8f0;
    margin-top: 10px;
}
"""

# ── JS: Enter key fix for Colab iframe ───────────────────────────────────────
ENTER_KEY_JS = """
() => {
    const bindEnterKey = () => {
        const textareas = document.querySelectorAll('textarea');
        textareas.forEach(ta => {
            if (!ta.dataset.enterBound) {
                ta.dataset.enterBound = 'true';
                ta.addEventListener('keydown', function(e) {
                    if (e.key === 'Enter' && !e.shiftKey) {
                        e.preventDefault();
                        const buttons = document.querySelectorAll('button');
                        for (const btn of buttons) {
                            if (btn.innerText.trim().includes('Send')) {
                                btn.click();
                                break;
                            }
                        }
                    }
                });
            }
        });
    };
    bindEnterKey();
    const retry = setInterval(bindEnterKey, 800);
    setTimeout(() => clearInterval(retry), 12000);
}
"""

# ── Build Gradio Layout ───────────────────────────────────────────────────────
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="indigo",
        neutral_hue="slate",
        font=gr.themes.GoogleFont("Inter"),
    ),
    css=CSS,
    title="Smart Contract Q&A Assistant",
) as demo:

    # ── Header ────────────────────────────────────────────────────────────────
    gr.HTML("""
    <div class="header-box">
      <h1 style="color:#f1f5f9;margin:0 0 8px 0;font-size:1.85em;
                 font-weight:700;letter-spacing:-0.4px;">
        🔐 Smart Contract Q&amp;A Assistant
      </h1>
      <p style="color:#94a3b8;margin:0;font-size:0.9em;letter-spacing:0.15px;">
        AI-powered legal &amp; technical document analysis &nbsp;·&nbsp;
        <span style="color:#60a5fa;font-weight:600;">Gemini</span> &nbsp;·&nbsp;
        <span style="color:#34d399;font-weight:600;">ChromaDB</span> &nbsp;·&nbsp;
        <span style="color:#c084fc;font-weight:600;">LangChain</span> &nbsp;·&nbsp;
        <span style="color:#fb923c;font-weight:600;">LangServe</span> &nbsp;·&nbsp;
        <span style="color:#f87171;font-weight:600;">Gradio</span>
      </p>
    </div>
    """)

    with gr.Row(equal_height=False):

        # ── Left Panel ────────────────────────────────────────────────────────
        with gr.Column(scale=1, min_width=290, elem_classes=["panel-card"]):

            gr.Markdown("## 📁 Documents")

            file_uploader = gr.File(
                label="Upload Contracts (PDF / DOCX / TXT)",
                file_count="multiple",
                file_types=[".pdf", ".docx", ".txt"],
            )
            upload_btn = gr.Button(
                "📥  Index Uploaded Files",
                variant="primary",
                size="lg",
            )
            with gr.Row():
                sample_btn = gr.Button("📋  Load Sample", variant="secondary", size="sm")
                clear_btn  = gr.Button("🗑️  Clear DB",    variant="stop",      size="sm")

            status_box = gr.Markdown(
                "*Upload a contract or load the sample to begin.*",
                elem_classes=["status-md"],
            )

            gr.Markdown("---")
            gr.Markdown(
                "**🔌 LangServe API — port 8001**\n"
                "```\n"
                "POST /contract-qa/invoke\n"
                "POST /contract-qa/batch\n"
                "POST /contract-qa/stream\n"
                "GET  /contract-qa/docs\n"
                "```"
            )
            gr.Markdown("---")
            gr.Markdown(
                "**💡 Example Questions:**\n\n"
                "- What is the total token supply?\n"
                "- What are the vesting terms for founders?\n"
                "- What happens if the soft cap is not reached?\n"
                "- What governance rights do token holders have?\n"
                "- Which jurisdictions are restricted?\n"
                "- Summarize the staking rewards and penalties."
            )

        # ── Right Panel ───────────────────────────────────────────────────────
        with gr.Column(scale=2, elem_classes=["panel-card"]):

            gr.Markdown("## 💬 Ask About Your Contract")

            chatbot = gr.Chatbot(
                value=[],
                label="",
                height=490,
                show_copy_button=True,
                type="messages",
                bubble_full_width=False,
                render_markdown=True,
                avatar_images=(
                    None,
                    "https://em-content.zobj.net/source/twitter/376/robot_1f916.png",
                ),
            )

            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="Type your question and press Enter ↵  or click  Send ➤",
                    lines=2,
                    scale=5,
                    show_label=False,
                    autofocus=True,
                )
                send_btn = gr.Button(
                    "Send ➤",
                    variant="primary",
                    scale=1,
                    size="lg",
                    min_width=110,
                )

            clear_chat_btn = gr.Button("🔄  Clear Conversation", size="sm")

    # ── Footer ────────────────────────────────────────────────────────────────
    gr.HTML("""
    <div class="footer-box">
      🛡️ Powered by
      <strong style="color:#2563eb;">Gemini API</strong> ·
      <strong style="color:#059669;">ChromaDB</strong> ·
      <strong style="color:#7c3aed;">LangChain</strong> ·
      <strong style="color:#d97706;">LangServe</strong> ·
      <strong style="color:#dc2626;">Gradio</strong><br>
      <span style="color:#94a3b8;font-size:0.95em;">
        Answers are grounded solely in the uploaded contract.
        Always verify critical terms with qualified legal counsel.
      </span>
    </div>
    """)

    # ── Event Wiring ──────────────────────────────────────────────────────────
    upload_btn.click(handle_upload,        [file_uploader], [status_box])
    sample_btn.click(load_sample_contract, [],              [status_box])
    clear_btn.click( clear_database,       [],              [status_box])

    send_btn.click(chat, [msg_input, chatbot], [chatbot, msg_input])

    demo.load(fn=None, js=ENTER_KEY_JS)

    clear_chat_btn.click(lambda: ([], ""), [], [chatbot, msg_input])


print("✅ Gradio interface built successfully!")
print("Launching — public share link will appear below...")
demo.queue()
demo.launch(share=True, debug=False, show_error=True, quiet=True, max_threads=10)

✅ gradio_client patches applied.
✅ LangServe already running — skipped.
✅ Gradio interface built successfully!
Launching — public share link will appear below...
* Running on public URL: https://c86a7bb42243b43783.gradio.live
